# 02 — BM25 Okapi Baseline

Establishes the lexical retrieval ceiling.
BM25 will be our primary baseline to beat with dense retrieval.

Key insight: BM25 fails on vocabulary mismatch cases.
Dense retrieval is designed to bridge this gap.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
from pathlib import Path

from src.bm25_retriever import BM25Retriever
from src.metrics import compute_metrics, aggregate, print_metrics_table

plt.style.use('seaborn-v0_8-whitegrid')
Path('../results').mkdir(parents=True, exist_ok=True)
print('Libraries loaded.')

In [ ]:
# Load corpus and test set
corpus_df = pd.read_parquet('../data/corpus.parquet')
test_df   = pd.read_parquet('../data/test.parquet')

corpus_ids  = corpus_df['product_id'].tolist()
corpus_docs = corpus_df['product_doc'].tolist()

print(f'Corpus: {len(corpus_df):,} products')
print(f'Test:   {len(test_df):,} queries')

In [ ]:
# Build BM25 index
bm25 = BM25Retriever(corpus_ids, corpus_docs)
print(f'BM25 index built over {bm25.corpus_size:,} products')

In [ ]:
# Demo retrieval
demo_query = test_df.iloc[0]['review_text']
demo_pid   = test_df.iloc[0]['product_id']

results = bm25.retrieve(demo_query, k=5)
true_doc = corpus_df[corpus_df['product_id']==demo_pid]['product_doc'].values[0] if demo_pid in corpus_ids else 'N/A'

print(f'Query: "{demo_query[:150]}"')
print(f'True product: "{true_doc[:100]}"')
print(f'\nBM25 Top-5 Results:')
for rank, (pid, score) in enumerate(results, 1):
    doc = corpus_df[corpus_df['product_id']==pid]['product_doc'].values
    doc_str = doc[0][:80] if len(doc) > 0 else 'N/A'
    hit = '✓' if pid == demo_pid else ' '
    print(f'  {hit} Rank {rank} (score={score:.2f}): {doc_str}')

In [ ]:
# Full test set evaluation
print('Running BM25 evaluation on full test set...')

per_query_metrics = []
for i, row in test_df.iterrows():
    results = bm25.retrieve(row['review_text'], k=10)
    retrieved_ids = [pid for pid, _ in results]
    metrics = compute_metrics(retrieved_ids, row['product_id'])
    per_query_metrics.append(metrics)
    
    if (i+1) % 500 == 0:
        print(f'  Processed {i+1:,}/{len(test_df):,} queries...')

bm25_metrics = aggregate(per_query_metrics)
print_metrics_table(bm25_metrics, title='BM25 Okapi Results')

In [ ]:
# NDCG@10 distribution
ndcg_scores = [m['ndcg@10'] for m in per_query_metrics]

fig, ax = plt.subplots(figsize=(10, 5))
# Only 0 or 1/log2(rank+1) — discrete distribution
ax.hist(ndcg_scores, bins=30, color='steelblue', alpha=0.8, edgecolor='white')
ax.axvline(np.mean(ndcg_scores), color='red', linestyle='--', lw=2,
           label=f'Mean NDCG@10 = {np.mean(ndcg_scores):.4f}')
ax.set_xlabel('NDCG@10', fontsize=12)
ax.set_ylabel('Number of Queries', fontsize=12)
ax.set_title('BM25 NDCG@10 Distribution (per query)', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig('../results/bm25_ndcg_dist.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Queries with NDCG@10=0 (complete failure): {sum(1 for x in ndcg_scores if x==0):,} ({100*sum(1 for x in ndcg_scores if x==0)/len(ndcg_scores):.1f}%)')
print(f'Queries with NDCG@10=1 (perfect rank 1):   {sum(1 for x in ndcg_scores if x==1.0):,} ({100*sum(1 for x in ndcg_scores if x==1.0)/len(ndcg_scores):.1f}%)')

In [ ]:
# Summary table
print('=' * 50)
print('BM25 Okapi — Final Results (Baseline)')
print('=' * 50)
for k, v in sorted(bm25_metrics.items()):
    print(f'  {k:<20}: {v:.4f}')
print('=' * 50)
print('\nThis is the lexical retrieval ceiling to beat.')
print('Dense retrieval wins on vocabulary mismatch cases.')